# Lecture 1 · Micrograd + MLP from scratch

Karpathy-style scalar autograd engine built in pure Python, then used to train a tiny MLP.

**Goal** — by the end you will have built a 100-line `Value` class that supports backprop, and trained a 2-layer MLP on a toy classification problem using only this engine.

**Prerequisites** · basic Python, chain rule from calculus.

---

## 1 · The `Value` class

A tiny scalar autograd: every number is a `Value`, and every operation builds a node in a computation graph.

In [ ]:
import math
import random

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f'Value(data={self.data:.4f}, grad={self.grad:.4f})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t*t) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    # Convenience ops
    def __neg__(self): return self * -1
    def __radd__(self, o): return self + o
    def __sub__(self, o): return self + (-o)
    def __rsub__(self, o): return (-self) + o
    def __rmul__(self, o): return self * o
    def __truediv__(self, o): return self * o**-1

print('Value class ready.')

## 2 · Smoke test

Let's verify autograd works on a simple expression.

In [ ]:
# f(x, y) = (x + y) * x · gradient at (3, 4) should be:
#   df/dx = 2x + y = 10
#   df/dy = x = 3
x = Value(3.0)
y = Value(4.0)
f = (x + y) * x
f.backward()
print(f'f = {f.data}')
print(f'df/dx = {x.grad}   (expected 10)')
print(f'df/dy = {y.grad}   (expected 3)')

## 3 · Neurons, layers, MLP

Build the standard module hierarchy using our `Value`.

In [ ]:
class Module:
    def parameters(self): return []
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

class Neuron(Module):
    def __init__(self, n_in):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self): return self.w + [self.b]

class Layer(Module):
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP(Module):
    def __init__(self, n_in, n_outs):
        sizes = [n_in] + n_outs
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(sizes) - 1)]

    def __call__(self, x):
        for l in self.layers:
            x = l(x)
        return x

    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

print('Module hierarchy ready.')

## 4 · Train on a toy binary task

Simple 4-example XOR-ish task. Watch the loss drop by hand.

In [ ]:
random.seed(42)

xs = [[2.0, 3.0, -1.0],
      [3.0, -1.0, 0.5],
      [0.5, 1.0, 1.0],
      [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]   # target output

model = MLP(3, [4, 4, 1])
print(f'Parameters: {len(model.parameters())}')

for step in range(100):
    # Forward: MSE
    ypreds = [model(x) for x in xs]
    loss = sum((yp - yt)**2 for yp, yt in zip(ypreds, ys))

    # Backward
    model.zero_grad()
    loss.backward()

    # SGD
    lr = 0.05
    for p in model.parameters():
        p.data -= lr * p.grad

    if step % 10 == 0:
        print(f'step {step:3d}  loss = {loss.data:.4f}')

print()
print('final predictions:')
for x, yt in zip(xs, ys):
    yp = model(x)
    print(f'  {x} → pred {yp.data:+.3f}  (target {yt:+.1f})')

## 5 · Now the same thing in PyTorch

Verify our hand-written autograd agrees with a real framework.

In [ ]:
import torch
import torch.nn as nn

X = torch.tensor(xs, dtype=torch.float32)
Y = torch.tensor(ys, dtype=torch.float32).unsqueeze(1)

torch_model = nn.Sequential(
    nn.Linear(3, 4), nn.Tanh(),
    nn.Linear(4, 4), nn.Tanh(),
    nn.Linear(4, 1), nn.Tanh(),
)
opt = torch.optim.SGD(torch_model.parameters(), lr=0.05)

for step in range(100):
    pred = torch_model(X)
    loss = ((pred - Y) ** 2).sum()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 10 == 0:
        print(f'step {step:3d}  loss = {loss.item():.4f}')

print()
print('final predictions (PyTorch):')
print(torch_model(X).detach().numpy().ravel())

## 6 · Reflection

- `Value.backward()` does a topological sort, then calls each node's `_backward` in reverse.
- Every op (`+`, `*`, `tanh`, etc.) stores a **local gradient rule** and a pointer to its children.
- PyTorch does the same thing, but with tensors (vectorized across batches) and C++ kernels.

**Next notebook** · build a real PyTorch MLP on MNIST and apply the debug ladder from L3.

**Reference** · Karpathy's *micrograd* and *makemore* videos on YouTube.